In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure the model
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
output = model.invoke("hi")
output.content

'Hi there! How can I help you today?'

In [2]:
# Configure the embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

d:\Krish Naik Youtube\Agentic AI Course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# data in vector database

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader_obj = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader)
docs = loader_obj.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)

new_docs=text_splitter.split_documents(docs)
doc_str = [doc.page_content for doc in new_docs]

In [ ]:
db = Chroma.from_documents(new_docs, embedding_model)

In [17]:
from pydantic import BaseModel, Field
from typing import TypedDict, Sequence, Annotated
from langchain_core.messages import BaseMessage
import operator

In [10]:
class TopicSelectionParser(BaseModel):
    Topic:str=Field(description="selected topic")
    Reasoning:str=Field(description="selected Reasoning behind topic selection")

In [14]:
from langchain.output_parsers import PydanticOutputParser

In [15]:
parser = PydanticOutputParser(pydantic_object=TopicSelectionParser)

In [16]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "selected Reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["Topic", "Reasoning"]}\n```'

In [ ]:
class AgentState(TypedDict):
    message: Annotated[Sequence[BaseMessage], operator.add]
    